In [ ]:

import pandas as pd

# 加载训练数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv'
train_df = pd.read_csv(train_data_path)

# 查看数据的基本信息
train_df.info()
train_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331 entries, 0 to 330
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       331 non-null    int64  
 1   gravity  331 non-null    float64
 2   ph       331 non-null    float64
 3   osmo     331 non-null    int64  
 4   cond     331 non-null    float64
 5   urea     331 non-null    int64  
 6   calc     331 non-null    float64
 7   target   331 non-null    int64  
dtypes: float64(4), int64(4)
memory usage: 20.8 KB


In [ ]:

import pandas as pd

# 加载训练数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv'
train_df = pd.read_csv(train_data_path)

# 查看数据的基本信息
train_df.info()
train_df.head()


Displayed on the user's machine.

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 重新加载数据以确保一致性
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv'
train_df = pd.read_csv(train_data_path)

# 检查缺失值
print(train_df.isnull().sum())

# 去除 'id' 列
train_df = train_df.drop(columns=['id'])

# 分离特征和目标变量
X = train_df.drop(columns=['target'])
y = train_df['target']

# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 将数据集分为训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 输出数据集的基本信息
print(f'Training set shape: {X_train.shape}')
print(f'Validation set shape: {X_val.shape}')


id         0
gravity    0
ph         0
osmo       0
cond       0
urea       0
calc       0
target     0
dtype: int64
Training set shape: (264, 6)
Validation set shape: (67, 6)


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 选择逻辑回归模型
model = LogisticRegression(random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 在验证集上进行预测
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# 计算验证集上的 AUC-ROC
val_auc_roc = roc_auc_score(y_val, y_val_pred_proba)
print(f'Validation AUC-ROC: {val_auc_roc}')



Validation AUC-ROC: 0.7843137254901961


In [ ]:


from sklearn.ensemble import RandomForestClassifier

# 选择随机森林模型
rf_model = RandomForestClassifier(random_state=42)

# 训练模型
rf_model.fit(X_train, y_train)

# 在验证集上进行预测
y_val_pred_proba_rf = rf_model.predict_proba(X_val)[:, 1]

# 计算验证集上的 AUC-ROC
val_auc_roc_rf = roc_auc_score(y_val, y_val_pred_proba_rf)
print(f'Validation AUC-ROC for Random Forest: {val_auc_roc_rf}')



Validation AUC-ROC for Random Forest: 0.8003565062388592


In [ ]:



from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 随机森林模型的超参数分布
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': [None] + list(range(10, 50, 10)),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['auto', 'sqrt', 'log2']
}

# 初始化随机搜索
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='roc_auc',
    random_state=42
)

# 执行随机搜索
random_search.fit(X_train, y_train)

# 最佳模型
best_rf_model = random_search.best_estimator_

# 在验证集上进行预测
y_val_pred_proba_best_rf = best_rf_model.predict_proba(X_val)[:, 1]

# 计算验证集上的 AUC-ROC
val_auc_roc_best_rf = roc_auc_score(y_val, y_val_pred_proba_best_rf)
print(f'Validation AUC-ROC for best Random Forest: {val_auc_roc_best_rf}')
print(f'Best parameters: {random_search.best_params_}')




Validation AUC-ROC for best Random Forest: 0.7932263814616756
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 58}
D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
24 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "D:\LLM-Driven_AI-Studio\MLAgent\

In [ ]:


from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 调整随机森林模型的超参数分布
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': [None] + list(range(10, 50, 10)),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}

# 初始化随机搜索
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='roc_auc',
    random_state=42
)

# 执行随机搜索
random_search.fit(X_train, y_train)

# 最佳模型
best_rf_model = random_search.best_estimator_

# 在验证集上进行预测
y_val_pred_proba_best_rf = best_rf_model.predict_proba(X_val)[:, 1]

# 计算验证集上的 AUC-ROC
val_auc_roc_best_rf = roc_auc_score(y_val, y_val_pred_proba_best_rf)
print(f'Validation AUC-ROC for best Random Forest: {val_auc_roc_best_rf}')
print(f'Best parameters: {random_search.best_params_}')


Validation AUC-ROC for best Random Forest: 0.7932263814616756
Best parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 58}


In [ ]:


# 加载测试数据
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/test.csv'
test_df = pd.read_csv(test_data_path)

# 去除 'id' 列
test_df = test_df.drop(columns=['id'])

# 标准化特征
X_test = scaler.transform(test_df)

# 在测试集上进行预测
y_test_pred_proba = best_rf_model.predict_proba(X_test)[:, 1]

# 计算测试集上的 AUC-ROC
test_auc_roc = roc_auc_score(y_test, y_test_pred_proba)
print(f'Test AUC-ROC: {test_auc_roc}')



---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[8], line 18
     16 pass
     17 print('##active_line9##')
---> 18 X_test = scaler.transform(test_df)
     19 print('##active_line10##')
     20 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_set_output.py:319, in _wrap_method_output.<locals>.wrapped(self, X, *args, **kwargs)
    317 @wraps(f)
    318 def wrapped(self, X, *args, **kwargs):
--> 319     data_to_wrap = f(self, X, *args, **kwargs)
    320     if isinstance(data_to_wrap, tuple):
    321         # only wrap the first output for cross decomposition
    322         return_tuple = (
    323             _wrap_data_with_container(method, data_to_wrap[0], X, self),
    324             *data_to_wrap[1:],
    325         )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\skle

In [ ]:


# 加载测试数据
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/test.csv'
test_df = pd.read_csv(test_data_path)

# 去除 'id' 和 'target' 列
test_df = test_df.drop(columns=['id', 'target'])

# 标准化特征
X_test = scaler.transform(test_df)

# 在测试集上进行预测
y_test_pred_proba = best_rf_model.predict_proba(X_test)[:, 1]

# 计算测试集上的 AUC-ROC
test_auc_roc = roc_auc_score(y_test, y_test_pred_proba)
print(f'Test AUC-ROC: {test_auc_roc}')




---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[9], line 30
     28 pass
     29 print('##active_line15##')
---> 30 test_auc_roc = roc_auc_score(y_test, y_test_pred_proba)
     31 print('##active_line16##')
     32 print(f'Test AUC-ROC: {test_auc_roc}')

NameError: name 'y_test' is not defined

In [ ]:



# 加载测试数据
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/test.csv'
test_df = pd.read_csv(test_data_path)

# 去除 'id' 列
test_df = test_df.drop(columns=['id'])

# 分离特征和目标变量
X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

# 标准化特征
X_test_scaled = scaler.transform(X_test)

# 在测试集上进行预测
y_test_pred_proba = best_rf_model.predict_proba(X_test_scaled)[:, 1]

# 计算测试集上的 AUC-ROC
test_auc_roc = roc_auc_score(y_test, y_test_pred_proba)
print(f'Test AUC-ROC: {test_auc_roc}')




Test AUC-ROC: 0.8064327485380116
